In [3]:
import os
import pandas as pd
import numpy as np

BASE = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
SRC = os.path.join(BASE, "Controversial_and_Clean_Holdings_final.xlsx")
OUT = os.path.join(BASE, "classification_binary.csv")

def read_all_sheets_xlsx(path):
    xls = pd.ExcelFile(path)
    frames = []
    for s in xls.sheet_names:
        df = pd.read_excel(xls, s)
        if isinstance(df, pd.DataFrame) and not df.empty:
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def pick(df, *cands):
    m = {c.lower().strip(): c for c in df.columns}
    for a in cands:
        if a in m:
            return m[a]
    for a in cands:
        for k in m:
            if a in k:
                return m[k]
    return None

df = read_all_sheets_xlsx(SRC)

c_etf_tic = pick(df, "etf ticker","etf_ticker","ticker")
c_etf_name = pick(df, "etf name","name")
c_screen = pick(df, "screen category","category","screen","tag","label")
c_co_tic = pick(df, "company_ticker","company ticker","ticker")
c_nm = pick(df, "name_normalized","normalised name","normalized name","name")
c_hold_nm = pick(df, "holding name","security","security name","name","name_normalized","normalised name")

df = df.rename(columns={
    c_etf_tic: "ETF Ticker",
    c_etf_name: "ETF Name",
    c_screen: "Screen Category",
    c_co_tic: "company_ticker",
    c_nm: "name_normalized",
    c_hold_nm: "Holding Name"
})[["ETF Ticker","ETF Name","Holding Name","Screen Category","company_ticker","name_normalized"]]

for col in ["ETF Ticker","ETF Name","Holding Name","Screen Category","company_ticker","name_normalized"]:
    if col not in df.columns:
        df[col] = np.nan

df["Screen Category"] = df["Screen Category"].astype(str).str.strip()
df["company_ticker"] = df["company_ticker"].astype(str).str.strip()
df["name_normalized"] = df["name_normalized"].astype(str).str.strip()
df["Holding Name"] = df["Holding Name"].astype(str).str.strip()

def flag_contains(s, key):
    s = str(s).lower()
    return int(key in s)

df["clean200"] = df["Screen Category"].str.lower().str.contains("clean200").astype(int)
df["deforestation"] = df["Screen Category"].str.lower().str.contains("deforestation").astype(int)
df["fossil fuel"] = df["Screen Category"].str.lower().str.contains("fossil").astype(int)
df["prison"] = df["Screen Category"].str.lower().str.contains("prison").astype(int)
df["tobacco"] = df["Screen Category"].str.lower().str.contains("tobacco").astype(int)
df["weapons"] = df["Screen Category"].str.lower().str.contains("weapon").astype(int)

grp_keys = ["ETF Ticker","ETF Name","Holding Name","company_ticker","name_normalized"]

agg = (
    df.groupby(grp_keys, as_index=False)
      .agg({
          "Screen Category": lambda x: "; ".join(sorted({str(v).strip() for v in x if pd.notna(v) and str(v).strip() != ""})),
          "clean200": "max",
          "deforestation": "max",
          "fossil fuel": "max",
          "prison": "max",
          "tobacco": "max",
          "weapons": "max"
      })
)

cols = ["ETF Ticker","ETF Name","Holding Name","Screen Category","company_ticker","name_normalized",
        "clean200","deforestation","fossil fuel","prison","tobacco","weapons"]

agg = agg[cols]
agg.to_csv(OUT, index=False)
print(OUT)


/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data/classification_binary.csv
